In [25]:
##### simple Gen AI app using Langchain
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_TRACING_PROJECT_NAME'] = 'my_project'


In [26]:
## Data Scraping
from langchain_community.document_loaders import WebBaseLoader


In [27]:
loader = WebBaseLoader("https://www.zeliot.in/blog/why-managed-kafka-is-no-enough-for-a-complete-streaming-data-platforms")
docs = loader.load()


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
texts = text_splitter.split_documents(docs)

In [29]:
texts

[Document(metadata={'source': 'https://www.zeliot.in/blog/why-managed-kafka-is-no-enough-for-a-complete-streaming-data-platforms', 'title': 'Why Managed Kafka Isn’t Enough: The Case for Full Streaming Platforms ', 'description': 'Managed Kafka handles clusters, not applications. Learn why full streaming platforms like Condense are essential for real-time pipelines in production.', 'language': 'en-US'}, page_content='Why Managed Kafka Isn’t Enough: The Case for Full Streaming Platforms'),
 Document(metadata={'source': 'https://www.zeliot.in/blog/why-managed-kafka-is-no-enough-for-a-complete-streaming-data-platforms', 'title': 'Why Managed Kafka Isn’t Enough: The Case for Full Streaming Platforms ', 'description': 'Managed Kafka handles clusters, not applications. Learn why full streaming platforms like Condense are essential for real-time pipelines in production.', 'language': 'en-US'}, page_content='Why CondenseDevelopersCompanyResourcesCustomersPricingRequest a DemoTry For FreeWhy Con

In [30]:
# Embedding and Vector Store
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()


In [31]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(texts, embeddings)
vectorstore

In [32]:
result = vectorstore.similarity_search("What is Condense?")
result[0].page_content

'application logic, state management, orchestration, and observability around it. Teams often have to stitch together multiple tools, increasing operational complexity and slowing development. Full streaming platforms like Condense solve all these problems by unifying stream processing, deployment, observability, and governance in one place, allowing teams to deliver real-time business outcomes'

In [33]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

In [34]:
## Retrieval chain, Document chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Anser the following question based on the provided context.
<context>{context}</context>

""")

doc_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
doc_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnser the following question based on the provided context.\n<context>{context}</context>\n\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x1155b7fd0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10fe05fd0>, root_client=<openai.OpenAI object at 0x1155b7820>, root_async_client=<openai.AsyncOpenAI object at 0x1155b7ee0>, model_name='gpt-4o', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser(), kwargs={}, config={'

In [39]:
from langchain_core.documents import Document

doc_chain.invoke({"input": "how zeliot helps business", "context": [Document(page_content="In essence, Zeliot helps businesses move from raw data to instant, actionable intelligence for smarter operations and innovation.")]})


'What does Zeliot help businesses achieve?\n\nZeliot helps businesses transform raw data into instant, actionable intelligence, enabling smarter operations and fostering innovation.'

In [40]:
vectorstore

In [46]:
retriever = vectorstore.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=doc_chain
)


In [49]:
response = retrieval_chain.invoke({"input": "What is Condense?"})
response['answer']

'What is the main advantage of using Condense as a streaming platform?\n\nThe main advantage of using Condense as a streaming platform is that it unifies stream processing, deployment, observability, and governance in one place. This integration simplifies real-time data streaming, reduces operational complexity, and accelerates development by providing a fully managed, Kafka-native platform with built-in connectors and support for BYOC (Bring Your Own Cloud).'